# Download a model from Hugging face

In [ ]:
%pip install -q boto3 huggingface_hub

## Download lightonai/LightOnOCR-2-1B

In [1]:
import os
import boto3
import urllib3
from huggingface_hub import snapshot_download

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def _env(name: str) -> str:
    v = os.environ.get(name, "").strip()
    if not v:
        raise SystemExit(f"Missing {name}. Mount minio + huggingface-token (envFrom).")
    return v


hf_token = _env("HF_TOKEN")
if hf_token in ("<YOUR_HF_TOKEN>", "changeme"):
    raise SystemExit("Set HF_TOKEN.")

aws_key = _env("AWS_ACCESS_KEY_ID")
aws_secret = _env("AWS_SECRET_ACCESS_KEY")
endpoint = _env("AWS_S3_ENDPOINT")
bucket_name = (os.environ.get("AWS_S3_BUCKET") or "models").strip()
verify_ssl = os.environ.get("S3_VERIFY_SSL", "false").lower() in ("1", "true", "yes")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

model_id = os.environ.get("MODEL_ID", "lightonai/LightOnOCR-2-1B")
s3_folder = os.environ.get("S3_FOLDER", "lighton-ocr")

print(f"--- Downloading {model_id} ---")
local_path = snapshot_download(
    repo_id=model_id,
    token=hf_token,
    # Include chat_template.jinja; required for vLLM chat (transformers 4.44+).
    allow_patterns=["*.json", "*.bin", "*.safetensors", "*.py", "*.txt", "*.jinja", "*.jinja2"],
    ignore_patterns=["*.msgpack", "*.h5"],
)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=aws_key,
    aws_secret_access_key=aws_secret,
    verify=verify_ssl,
)
try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Created bucket: {bucket_name}")
except Exception:
    pass

print(f"--- Uploading to {bucket_name}/{s3_folder} ---")
for root, _, files in os.walk(local_path):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, local_path)
        s3_key = f"{s3_folder}/{rel_path}"
        print(f"Uploading {rel_path}...")
        s3.upload_file(full_path, bucket_name, s3_key)

print("\nDONE:", f"s3://{bucket_name}/{s3_folder}/")

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Downloading lightonai/LightOnOCR-2-1B ---


Fetching 9 files: 100%|██████████| 9/9 [00:19<00:00,  2.17s/it]


Created bucket: models
--- Uploading to models/lighton-ocr ---
Uploading chat_template.jinja...
Uploading tokenizer.json...
Uploading config.json...
Uploading generation_config.json...
Uploading model.safetensors...
Uploading processor_config.json...
Uploading tokenizer_config.json...
Uploading added_tokens.json...
Uploading special_tokens_map.json...

DONE: s3://models/lighton-ocr/


## Download openai/gpt-oss-20b

In [4]:
import os
import boto3
import urllib3
from huggingface_hub import snapshot_download

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def _env(name: str) -> str:
    v = os.environ.get(name, "").strip()
    if not v:
        raise SystemExit(
            f"Missing {name}. Add minio + huggingface-token data connections (envFrom) in the workbench."
        )
    return v


hf_token = _env("HF_TOKEN")
if hf_token in ("<YOUR_HF_TOKEN>", "changeme"):
    raise SystemExit("Set HF_TOKEN (huggingface-token secret).")

aws_key = _env("AWS_ACCESS_KEY_ID")
aws_secret = _env("AWS_SECRET_ACCESS_KEY")
endpoint = _env("AWS_S3_ENDPOINT")
bucket_name = (os.environ.get("AWS_S3_BUCKET") or "models").strip()
verify_ssl = os.environ.get("S3_VERIFY_SSL", "false").lower() in ("1", "true", "yes")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

model_id = os.environ.get("MODEL_ID", "openai/gpt-oss-20b")
s3_folder = os.environ.get("S3_FOLDER", "gpt-oss-20b")

print(f"--- Downloading {model_id} ---")
local_path = snapshot_download(
    repo_id=model_id,
    token=hf_token,
    allow_patterns=["*.json", "*.bin", "*.safetensors", "*.py", "*.txt"],
    ignore_patterns=["*.msgpack", "*.h5"],
)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=aws_key,
    aws_secret_access_key=aws_secret,
    verify=verify_ssl,
)
try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Created bucket: {bucket_name}")
except Exception:
    pass

print(f"--- Uploading to {bucket_name}/{s3_folder} ---")
for root, _, files in os.walk(local_path):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, local_path)
        s3_key = f"{s3_folder}/{rel_path}"
        print(f"Uploading {rel_path}...")
        s3.upload_file(full_path, bucket_name, s3_key)

print("\nDONE:", f"s3://{bucket_name}/{s3_folder}/")

--- Downloading openai/gpt-oss-20b ---


Fetching 13 files: 100%|██████████| 13/13 [05:18<00:00, 24.53s/it]


--- Uploading to models/gpt-oss-20b ---
Uploading model-00001-of-00002.safetensors...
Uploading tokenizer.json...
Uploading model.safetensors.index.json...
Uploading config.json...
Uploading model-00000-of-00002.safetensors...
Uploading generation_config.json...
Uploading tokenizer_config.json...
Uploading model-00002-of-00002.safetensors...
Uploading special_tokens_map.json...
Uploading original/config.json...
Uploading original/model.safetensors...
Uploading original/dtypes.json...
Uploading metal/model.bin...

DONE: s3://models/gpt-oss-20b/


## Check contents of bucket 'models'

In [5]:
# Check contents of bucket
import os
import boto3

# From env (same names as the MinIO / OpenShift connection secret)
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ.get("AWS_S3_ENDPOINT"),  # e.g. http://minio-service.minio.svc:9000
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

bucket = os.environ.get("AWS_S3_BUCKET", "models")
prefix = ""  # optional, or "" for the whole bucket

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        print(f"{obj['Key']}\t{obj['Size']}")

gpt-oss-20b/config.json	1806
gpt-oss-20b/generation_config.json	177
gpt-oss-20b/metal/model.bin	13750886400
gpt-oss-20b/model-00000-of-00002.safetensors	4792272488
gpt-oss-20b/model-00001-of-00002.safetensors	4798702184
gpt-oss-20b/model-00002-of-00002.safetensors	4170342232
gpt-oss-20b/model.safetensors.index.json	36355
gpt-oss-20b/original/config.json	376
gpt-oss-20b/original/dtypes.json	13082
gpt-oss-20b/original/model.safetensors	13761300984
gpt-oss-20b/special_tokens_map.json	98
gpt-oss-20b/tokenizer.json	27868174
gpt-oss-20b/tokenizer_config.json	4200
lighton-ocr/added_tokens.json	707
lighton-ocr/chat_template.jinja	720
lighton-ocr/config.json	2363
lighton-ocr/generation_config.json	219
lighton-ocr/model.safetensors	2011367489
lighton-ocr/processor_config.json	1023
lighton-ocr/special_tokens_map.json	613
lighton-ocr/tokenizer.json	11422822
lighton-ocr/tokenizer_config.json	5562
